Chi-square goodness-of-fit test

Below is a practical Chi-square goodness-of-fit test using scipy.stats.chisquare(), with hypothesis setup, decision making, and what to do after the decision.

Finance example

Suppose a trading strategy has 1,000 trades categorized into four outcomes. Based on your assumed/benchmark distribution, you expect:

Large loss: 10%
Small loss: 30%
Small gain: 40%
Large gain: 20%

Observed results:

Large loss: 120
Small loss: 280
Small gain: 420
Large gain: 180

We want to test:
H0​:Observed distribution follows the expected distribution
H1​:Observed distribution differs from the expected distribution

In [1]:
import numpy as np
from scipy.stats import chisquare


# --------------------------------------------------
# 1. Observed frequencies
# --------------------------------------------------

observed = np.array([
    120,   # Large loss
    280,   # Small loss
    420,   # Small gain
    180    # Large gain
])


# --------------------------------------------------
# 2. Expected probabilities under H0
# --------------------------------------------------

expected_probabilities = np.array([
    0.10,  # Large loss
    0.30,  # Small loss
    0.40,  # Small gain
    0.20   # Large gain
])


# --------------------------------------------------
# 3. Convert probabilities into expected counts
# --------------------------------------------------

n = observed.sum()

expected = n * expected_probabilities


# --------------------------------------------------
# 4. Chi-square goodness-of-fit test
# --------------------------------------------------

result = chisquare(
    f_obs=observed,
    f_exp=expected
)


chi_square_statistic = result.statistic
p_value = result.pvalue


# --------------------------------------------------
# 5. Decision
# --------------------------------------------------

alpha = 0.05

if p_value < alpha:
    decision = "Reject H0"
    conclusion = (
        "There is statistically significant evidence that "
        "the observed distribution differs from the expected distribution."
    )
else:
    decision = "Fail to reject H0"
    conclusion = (
        "There is insufficient statistical evidence to conclude "
        "that the observed distribution differs from the expected distribution."
    )


# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("=" * 60)
print("CHI-SQUARE GOODNESS-OF-FIT TEST")
print("=" * 60)

print(f"Total observations : {n}")
print(f"Observed counts    : {observed}")
print(f"Expected counts    : {expected}")

print("-" * 60)

print(f"Chi-square statistic : {chi_square_statistic:.4f}")
print(f"P-value              : {p_value:.6f}")
print(f"Alpha                : {alpha}")

print("-" * 60)

print(f"Decision   : {decision}")
print(f"Conclusion : {conclusion}")

print("=" * 60)

CHI-SQUARE GOODNESS-OF-FIT TEST
Total observations : 1000
Observed counts    : [120 280 420 180]
Expected counts    : [100. 300. 400. 200.]
------------------------------------------------------------
Chi-square statistic : 8.3333
P-value              : 0.039602
Alpha                : 0.05
------------------------------------------------------------
Decision   : Reject H0
Conclusion : There is statistically significant evidence that the observed distribution differs from the expected distribution.


A reusable function

If you want to use this as part of your statistics/agentic-AI functions, make it reusable:

In [2]:
import numpy as np
from scipy.stats import chisquare


def chi_square_goodness_of_fit(
    observed,
    expected_probabilities,
    alpha=0.05
):
    """
    Perform a Chi-square goodness-of-fit test.

    Parameters
    ----------
    observed : array-like
        Observed frequencies/counts.

    expected_probabilities : array-like
        Expected probabilities under H0.
        They must sum to 1.

    alpha : float
        Significance level.

    Returns
    -------
    dict
        Test results and decision.
    """

    observed = np.asarray(observed, dtype=float)
    expected_probabilities = np.asarray(
        expected_probabilities,
        dtype=float
    )

    # Validate input
    if len(observed) != len(expected_probabilities):
        raise ValueError(
            "Observed and expected probabilities "
            "must have the same number of categories."
        )

    if not np.isclose(expected_probabilities.sum(), 1):
        raise ValueError(
            "Expected probabilities must sum to 1."
        )

    if np.any(observed < 0):
        raise ValueError(
            "Observed counts cannot be negative."
        )

    # Total observations
    n = observed.sum()

    # Expected frequencies
    expected = n * expected_probabilities

    # Chi-square test
    result = chisquare(
        f_obs=observed,
        f_exp=expected
    )

    chi_square_statistic = result.statistic
    p_value = result.pvalue

    # Decision
    if p_value < alpha:
        decision = "Reject H0"
        conclusion = (
            "There is statistically significant evidence "
            "that the observed distribution differs "
            "from the expected distribution."
        )
    else:
        decision = "Fail to reject H0"
        conclusion = (
            "There is insufficient statistical evidence "
            "to conclude that the observed distribution "
            "differs from the expected distribution."
        )

    return {
        "total_observations": int(n),
        "observed": observed,
        "expected": expected,
        "chi_square_statistic": chi_square_statistic,
        "p_value": p_value,
        "alpha": alpha,
        "decision": decision,
        "conclusion": conclusion
    }

Use it like this:

In [4]:
observed = [120, 280, 420, 180]

expected_probabilities = [
    0.10,
    0.30,
    0.40,
    0.20
]

result = chi_square_goodness_of_fit(
    observed=observed,
    expected_probabilities=expected_probabilities,
    alpha=0.05
)

print("Chi-square statistic:",
      result["chi_square_statistic"])

print("P-value:",
      result["p_value"])

print("Decision:",
      result["decision"])

print("Conclusion:",
      result["conclusion"])

Chi-square statistic: 8.333333333333332
P-value: 0.03960235520756418
Decision: Reject H0
Conclusion: There is statistically significant evidence that the observed distribution differs from the expected distribution.


This tells you which categories contribute most to the overall chi-square statistic.

In [5]:
contributions = (observed - expected) ** 2 / expected

for i, contribution in enumerate(contributions):
    print(
        f"Category {i+1}: "
        f"{contribution:.4f}"
    )

Category 1: 4.0000
Category 2: 1.3333
Category 3: 1.0000
Category 4: 2.0000
